# FRED ingestion - orchestrator

Iterates over the configured series, calling the worker notebook once per series via `dbutils.notebook.run`, then appends every result to `finhive.logs.ingestionLog`.

In [ ]:
import os

dbutils.widgets.text("job_name", "")
dbutils.widgets.text("config_path", "")

job_name = dbutils.widgets.get("job_name")
config_path = dbutils.widgets.get("config_path")

if not config_path:
    notebook_path = (
        dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        .notebookPath().get()
    )
    repo_root = "/Workspace" + notebook_path.rsplit("/notebooks/", 1)[0]
    config_path = f"{repo_root}/config/data_ingestion/fred.json"

if not job_name:
    raise ValueError("widget 'job_name' is required")

In [ ]:
dbutils.widgets.text("worker_notebook_path", "./worker")
worker_notebook_path = dbutils.widgets.get("worker_notebook_path")

In [ ]:
import json

with open(config_path) as f:
    series_config = json.load(f)

if not series_config:
    raise ValueError(f"no series configured in {config_path}")

In [ ]:
results = []

for entry in series_config:
    series = entry["series"]
    start_date = entry.get("start_date")

    raw_result = dbutils.notebook.run(
        worker_notebook_path,
        timeout_seconds=3600,
        arguments={
            "series": series,
            "start_date": start_date or "",
            "job_name": job_name,
        },
    )
    result = json.loads(raw_result)
    results.append(result)

    if result["status"]:
        entry["start_date"] = result["updateAt"]

In [ ]:
with open(config_path, "w") as f:
    json.dump(series_config, f, indent=4)

In [ ]:
from datetime import datetime

spark.sql("CREATE CATALOG IF NOT EXISTS finhive")
spark.sql("CREATE SCHEMA IF NOT EXISTS finhive.logs")

log_rows = [
    (
        r["series"],
        r["source"],
        r["status"],
        datetime.fromisoformat(r["updateAt"]),
        r["item_count"],
        r["error"],
        r["job_name"],
    )
    for r in results
]
log_schema = "series string, source string, status boolean, updateAt timestamp, item_count int, error string, job_name string"
log_df = spark.createDataFrame(log_rows, schema=log_schema)
log_df.write.mode("append").saveAsTable("finhive.logs.ingestionLog")

In [ ]:
failures = [r for r in results if not r["status"]]
if failures:
    failed_series = [r["series"] for r in failures]
    raise RuntimeError(f"{len(failures)} series failed to ingest: {failed_series}")

print(f"ingested {len(results)} series successfully")